# EtherNet/IP — objek CIP dan tag Logix

HMI (`EnipClient`) berbicara dengan PLC lini pengemasan simulasi (`EnipAdapter`) lewat koneksi TCP dalam memori. Arahkan klien ke pengendali dengan `UseTcp("192.168.1.20", 44818).UseSlot(0)`; klien tetap hanya-baca sampai `AllowWrites()`.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.30.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.30.0-preview.1"

## Permintaan dan path CIP
Permintaan adalah layanan, EPATH, dan data; segmen simbolik menamai tag Logix.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Protocols.EtherNetIp;
using IoTCom.Net.Transports;

var name = new CipRequest(CipService.GetAttributeSingle, CipPath.Logical(CipClass.Identity, 1, 7), []);
Console.WriteLine($"{name}: {Convert.ToHexString(name.Encode())}");
var read = new CipRequest(CipService.ReadTag, CipPath.Tag("Zone_Temps[2]"), [1, 0]);
Console.WriteLine($"{read}: {Convert.ToHexString(read.Encode())}");
var packet = new EnipPacket { Command = EnipCommand.SendRRData, Session = 0x1001, Data = EnipPacket.SendRRData(read.Encode()) }.Encode();
foreach (var f in EnipPacket.DescribeFields(packet)) Console.WriteLine($"  {f.Name,-18} {f.Value}");

## Sesi, identitas, dan tag
RegisterSession, objek Identity, dan delapan tag dalam satu Multiple_Service_Packet.

In [ ]:
var plcListener = new InMemoryTransportListener("packline");
var plc = EnipPlcSimulator.Create(o => o.ListenInMemory(plcListener));
await plc.StartAsync();
var hmi = EnipClient.Create(o => o.UseInMemory(plcListener).UseSlot(0));
await hmi.ConnectAsync();
Console.WriteLine($"session 0x{hmi.Session:X8}: {await hmi.ReadIdentityAsync()}");
foreach (var (tag, value, error) in await hmi.ReadTagsAsync(["Line_Speed", "Motor_Run", "Count_Good", "Count_Reject", "Seal_Temp", "Missing"]))
    Console.WriteLine($"  {tag,-13} {value?.ToString() ?? error!.Status.ToString()}");
Console.WriteLine($"  Zone_Temps    {await hmi.ReadTagAsync("Zone_Temps", 4)}");

## Penulisan harus diizinkan
HMI ini hanya-baca; HMI kedua dengan AllowWrites() mendinginkan seal jaw, dan reject naik.

In [ ]:
try { await hmi.WriteTagAsync("Seal_Setpoint", CipTagValue.Of(CipDataType.Real, 160)); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }
var engineer = EnipClient.Create(o => o.UseInMemory(plcListener).AllowWrites());
await engineer.ConnectAsync();
var before = (await hmi.ReadTagAsync("Count_Reject")).Value;
await engineer.WriteTagAsync("Seal_Setpoint", CipTagValue.Of(CipDataType.Real, 160));
for (var i = 0; i < 10; i++) plc.Step(30);
Console.WriteLine($"rejects +{(await hmi.ReadTagAsync("Count_Reject")).Value - before} in five minutes at a 160 °C seal");
try { await engineer.WriteTagAsync("Count_Good", CipTagValue.Of(CipDataType.Dint, 0)); } catch (CipException e) { Console.WriteLine($"Count_Good: {e.Status}"); }
await engineer.DisposeAsync();
await hmi.DisposeAsync();
await plc.DisposeAsync();

## Lebih lanjut
`iotcom enip demo`, `iotcom enip discover`, `iotcom enip read <tag> --host <plc> --slot 0`, `iotcom enip serve`, `iotcom enip decode <hex>`, *Lini pengemasan lewat EtherNet/IP* di Gallery, dan sampel EnipTags. Lihat `docs/id/protocols/ethernetip.md`.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*